# PySpark Practice — Retail Orders

Same data as `excel_practice.xlsx`. Where an exercise has an Excel twin, the number is noted —
**your Spark answer must match the Excel answer exactly.** That cross-check is the whole point:
it catches the silent mistakes (dirty joins, dropped nulls, wrong dedup key) that both tools
let you make quietly.

**Setup:** put `retail_orders.csv`, `dim_products.csv`, `dim_reps.csv`, `dim_targets.csv`
in the same folder as this notebook.

Rule for yourself: attempt each cell before scrolling to the solutions at the bottom.

In [ ]:
from pyspark.sql import SparkSession, functions as F, Window
from pyspark.sql.types import *

spark = (SparkSession.builder
         .appName("probation-practice")
         .master("local[*]")
         .config("spark.sql.shuffle.partitions", "8")   # 200 is silly on a laptop
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")
print(spark.version)

## Block 0 — Reading data properly

Assessments love `inferSchema`. Know why it is a bad default: it triggers a **full extra pass**
over the file. An explicit schema is one pass and it is deterministic.

In [ ]:
# Q0.1 — read with inferSchema, then TIME it
import time
t = time.time()
df_lazy = spark.read.csv("retail_orders.csv", header=True, inferSchema=True)
print("infer:", round(time.time()-t, 2), "s")
df_lazy.printSchema()

In [ ]:
# Q0.2 — now define the schema explicitly and read again. Compare timing.
schema = StructType([
    StructField("OrderID", StringType(), True),
    # TODO: fill in the remaining 12 fields with correct types.
    # OrderDate should be DateType, Units IntegerType, UnitPrice/DiscountPct DoubleType,
    # ShipDays DoubleType (it has nulls), the rest StringType.
])

# orders = spark.read.csv("retail_orders.csv", header=True, schema=schema)
# print(orders.count())

In [ ]:
# Q0.3 — What does .count() trigger? What did .read.csv() trigger?
# Write your answer here as a comment, then verify in the Spark UI at http://localhost:4040
# (look at the Jobs tab — how many jobs appeared, and after which line?)


## Block 1 — Cleaning

Mirrors Excel Block 1. Same defects: dirty `Region` casing/whitespace, null `ShipDays`,
negative `Units`, exact duplicate rows.

In [ ]:
# Q1.1 — clean Region: trim whitespace, normalise casing. Show distinct values before & after.
# Expect 4 clean values at the end.


In [ ]:
# Q1.2 — count rows with negative Units.   [Excel Q2 — expect 40]


In [ ]:
# Q1.3 — count null ShipDays.   [Excel Q3 — expect 120]
# Then fill nulls with the median ShipDays. Why median and not mean here?


In [ ]:
# Q1.4 — the file has exact duplicate rows.
#   (a) count rows involved in a duplicate OrderID   [Excel Q4 — expect 90]
#   (b) drop duplicates. Does dropDuplicates() on all columns give the same count
#       as dropDuplicates(["OrderID"])? Why might they differ in general?


In [ ]:
# Q1.5 — add NetRevenue = Units * UnitPrice * (1 - DiscountPct)
#   [Excel Q5 — total should be 472,566,490.35 on the RAW data, dupes included]
# Match the Excel number first, THEN decide whether dedup is the right call.


## Block 2 — Joins

The single highest-value PySpark assessment topic. Know join types cold, and know
what a broadcast does.

In [ ]:
# Q2.1 — join products onto orders on ProductID. Inner join.
# Before running: predict the row count. Then check. Were you right?


In [ ]:
# Q2.2 — join reps as well. Both dims are tiny (55 and 20 rows).
# Force a broadcast join explicitly, then run .explain() on both the plain and
# broadcast versions. Find BroadcastHashJoin vs SortMergeJoin in the plans.


In [ ]:
# Q2.3 — deliberately create a duplicate key in dim_products and re-join.
# Watch the row count explode. This is join fan-out — the bug that silently
# inflates every downstream aggregate. How would you detect it in production?


In [ ]:
# Q2.4 — left join vs inner: are there any ProductIDs in orders with no match in products?
# Prove it with a left_anti join.


## Block 3 — Aggregation

Direct Excel twins. Numbers must match.

In [ ]:
# Q3.1 — net revenue for West region.   [Excel Q14 — expect 112,474,613.39]
# Careful: use the CLEANED region column or you will undercount.


In [ ]:
# Q3.2 — net revenue: Online channel + Corporate segment + year 2025.
#   [Excel Q15 — expect 25,853,709.12]


In [ ]:
# Q3.3 — average units per order, Electronics category, Returned == 'No'.
#   [Excel Q16 — expect 3.5214]


In [ ]:
# Q3.4 — distinct rep count per region.   [Excel Q17 — South expects 5]
# Use countDistinct. Then look up approx_count_distinct — when would you prefer it?


In [ ]:
# Q3.5 — max single-order net revenue per region.   [Excel Q18 — North expects 791,951.89]


In [ ]:
# Q3.6 — one agg() call producing, per Region+Year:
#   total revenue, order count, avg discount, distinct products, return rate.
# Then pivot it: Region as rows, Year as columns, revenue as values.
# (df.groupBy(...).pivot(...) — this IS a pivot table, note the parallel to Excel.)


## Block 4 — Window functions

Assessments test these heavily and they have no clean Excel equivalent,
so this block is pure Spark.

In [ ]:
# Q4.1 — rank reps by total revenue within their region.
# Use all three: row_number, rank, dense_rank. Explain how they differ on ties.


In [ ]:
# Q4.2 — top 3 products by revenue in each category.


In [ ]:
# Q4.3 — monthly revenue, plus a 3-month moving average.
# rowsBetween(-2, 0) — and be clear about the difference from rangeBetween.


In [ ]:
# Q4.4 — month-over-month growth % using lag().


In [ ]:
# Q4.5 — running cumulative revenue per region, ordered by date.


## Block 5 — Performance and internals

The conceptual questions. Answer in words, then verify in the Spark UI.

In [ ]:
# Q5.1 — classify each as NARROW or WIDE, then verify with .explain():
#   select, filter, withColumn, groupBy, join, orderBy, repartition, coalesce, distinct


In [ ]:
# Q5.2 — check current partition count. Repartition to 4, then coalesce to 2.
# Which one shuffles? Why can coalesce only reduce partitions?
print(orders.rdd.getNumPartitions()) if 'orders' in dir() else None

In [ ]:
# Q5.3 — cache a filtered DataFrame, run two actions, time both.
# Then try persist(StorageLevel.MEMORY_AND_DISK). When does cache() hurt?


In [ ]:
# Q5.4 — write the joined DataFrame as Parquet partitioned by Region.
# Inspect the folder structure. Read one region back — how many files did Spark touch?
# Explain partition pruning.


In [ ]:
# Q5.5 — same as Q3.6 but in Spark SQL via createOrReplaceTempView + spark.sql().
# Compare .explain() output between DataFrame API and SQL. What do you notice?


In [ ]:
# Q5.6 — why is .collect() dangerous? What are the safer alternatives,
# and what does spark.driver.maxResultSize protect against?


## Block 6 — UDFs and the last-resort rule

In [ ]:
# Q6.1 — write a Python UDF that buckets ShipDays into '1-3','4-6','7-9','10+'.
# Then rewrite it with F.when().otherwise() chains. Time both on the full dataset.


In [ ]:
# Q6.2 — rewrite it a third time as a pandas_udf. Explain the serialisation
# difference between a plain UDF and a pandas UDF (Arrow).


---
# Solutions

Scroll no further until you have attempted the block.

In [ ]:
# --- Setup used by the solutions ---
schema = StructType([
    StructField("OrderID", StringType()), StructField("OrderDate", DateType()),
    StructField("Region", StringType()), StructField("City", StringType()),
    StructField("RepID", StringType()), StructField("ProductID", StringType()),
    StructField("Channel", StringType()), StructField("CustomerSegment", StringType()),
    StructField("Units", IntegerType()), StructField("UnitPrice", DoubleType()),
    StructField("DiscountPct", DoubleType()), StructField("ShipDays", DoubleType()),
    StructField("Returned", StringType()),
])
orders = spark.read.csv("retail_orders.csv", header=True, schema=schema)
products = spark.read.csv("dim_products.csv", header=True, inferSchema=True)
reps = spark.read.csv("dim_reps.csv", header=True, inferSchema=True)

# Q1.1 clean region + Q1.5 net revenue
o = (orders
     .withColumn("RegionClean", F.initcap(F.trim(F.col("Region"))))
     .withColumn("NetRevenue", F.col("Units") * F.col("UnitPrice") * (1 - F.col("DiscountPct"))))
o.select("RegionClean").distinct().show()

# Q1.2 / Q1.3 / Q1.4
print("neg units:", o.filter(F.col("Units") < 0).count())
print("null ship:", o.filter(F.col("ShipDays").isNull()).count())
dup_ids = (o.groupBy("OrderID").count().filter("count > 1").select("OrderID"))
print("rows in dupes:", o.join(dup_ids, "OrderID", "inner").count())

# Q1.5
print("total net rev:", o.agg(F.sum("NetRevenue")).first()[0])

In [ ]:
# ============ BLOCK 2 SOLUTIONS — Joins ============

# Q2.1 — inner join on ProductID
oj = o.join(products, "ProductID", "inner")
print("orders:", o.count(), " after join:", oj.count())
# Both are 6045. Predicting this correctly matters: an inner join on a UNIQUE key in the
# right table cannot change the row count IF every left key matches. Two separate
# assumptions, and Q2.3 / Q2.4 test them one at a time.

# Note the join syntax. Three forms, and they are NOT equivalent:
#   o.join(products, "ProductID")                                  -> one merged key column
#   o.join(products, o.ProductID == products.ProductID)            -> TWO ProductID columns
#   o.join(products, ["ProductID", "X"])                           -> multi-key
# The string form dedupes the key. The expression form leaves both, and any later
# select("ProductID") then throws AMBIGUOUS_REFERENCE. Prefer the string form.


In [ ]:
# Q2.2 — add reps, force a broadcast, compare plans

# FIRST: a collision you must notice. Reps also has a "Region" column.
print([c for c in reps.columns if c in o.columns])   # ['RepID', 'Region']

# Joining blindly gives you two Region columns and an ambiguous reference later.
# Drop or rename before joining:
reps_clean = reps.drop("Region").withColumnRenamed("RepName", "SalesRep")

full = (o.join(F.broadcast(products), "ProductID", "inner")
         .join(F.broadcast(reps_clean), "RepID", "inner"))
print("rows:", full.count())     # still 6045

# Compare the plans
plain = o.join(products, "ProductID").join(reps_clean, "RepID")
print("=== WITHOUT broadcast hint ===")
plain.explain()
print("=== WITH broadcast hint ===")
full.explain()

# What to look for: BroadcastHashJoin + BroadcastExchange vs SortMergeJoin + two
# Exchange (hashpartitioning) nodes. SortMergeJoin shuffles BOTH sides across the
# cluster and sorts them; broadcast ships the small table to every executor and
# shuffles nothing.
#
# Assessment answer: Spark broadcasts automatically when the smaller side is under
# spark.sql.autoBroadcastJoinThreshold (default 10 MB), so at 55 and 20 rows these
# would broadcast even without the hint. The hint matters when Spark's size estimate
# is wrong -- typically after a filter or on a table with stale/absent statistics.
print(spark.conf.get("spark.sql.autoBroadcastJoinThreshold"))


In [ ]:
# Q2.3 — join fan-out, the silent aggregate-inflater

dirty = products.union(products.limit(3))     # 3 ProductIDs now appear twice
print("dim rows:", products.count(), "->", dirty.count())

blown = o.join(dirty, "ProductID", "inner")
print("orders:", o.count(), " after dirty join:", blown.count())   # 6384, not 6045

# 339 phantom rows. Nothing errored. Every downstream sum is now overstated:
print("clean revenue:", o.join(products, "ProductID").agg(F.sum("NetRevenue")).first()[0])
print("dirty revenue:", blown.agg(F.sum("NetRevenue")).first()[0])

# How to detect it in production -- three habits, in order of cost:
# 1. Assert the key is unique on the dimension side BEFORE joining:
dup_keys = dirty.groupBy("ProductID").count().filter("count > 1")
print("duplicate dim keys:", dup_keys.count())
assert dup_keys.count() == 0, "dimension key is not unique -- join will fan out"

# 2. Assert row count is preserved across a join you expect to be 1:1:
before = o.count()
after = o.join(products, "ProductID", "inner").count()
assert before == after, f"fan-out: {before} -> {after}"

# 3. In a real pipeline, make it a data-quality test on the dimension table itself
#    (Great Expectations / dbt tests / a Delta constraint) so it fails at the source
#    rather than in whichever downstream job happens to notice first.


In [ ]:
# Q2.4 — left_anti to prove referential integrity

orphans = o.join(products, "ProductID", "left_anti")
print("orders with no matching product:", orphans.count())    # 0
orphans.show(5)

orphan_reps = o.join(reps, "RepID", "left_anti")
print("orders with no matching rep:", orphan_reps.count())    # 0

# left_anti = rows on the left with NO match on the right. It returns ONLY left
# columns, so it is cheaper than a left join + isNull filter and it cannot fan out.
# Its mirror, left_semi, returns left rows that DO match -- again left columns only,
# and again no fan-out even when the right side has duplicate keys. That property is
# why left_semi is the correct tool for "filter A by membership in B".

# The equivalent-but-worse idiom, for recognition:
#   o.join(products, "ProductID", "left").filter(F.col("Category").isNull())
# Works, but drags every right-side column through the shuffle and breaks if
# Category is legitimately null in the dimension.

# Because both anti-joins return 0 here, the Q2.1 prediction was safe: every key
# matches AND the dimension key is unique. Verify both before trusting a row count.


In [ ]:
# Q3.1 / Q3.2
print(o.filter(F.col("RegionClean") == "West").agg(F.sum("NetRevenue")).first()[0])

print(o.filter((F.col("Channel") == "Online") &
               (F.col("CustomerSegment") == "Corporate") &
               (F.year("OrderDate") == 2025))
       .agg(F.sum("NetRevenue")).first()[0])

# Q3.3 — needs the product join for Category
oj = o.join(F.broadcast(products), "ProductID", "inner")
print(oj.filter((F.col("Category") == "Electronics") & (F.col("Returned") == "No"))
        .agg(F.avg("Units")).first()[0])

# Q3.4 / Q3.5
o.groupBy("RegionClean").agg(
    F.countDistinct("RepID").alias("reps"),
    F.max("NetRevenue").alias("max_order")
).orderBy("RegionClean").show()

In [ ]:
# Q3.6 — multi-metric agg, then pivot
summary = (oj.withColumn("Year", F.year("OrderDate"))
   .groupBy("RegionClean", "Year")
   .agg(F.round(F.sum("NetRevenue"), 2).alias("revenue"),
        F.count("*").alias("orders"),
        F.round(F.avg("DiscountPct"), 3).alias("avg_disc"),
        F.countDistinct("ProductID").alias("products"),
        F.round(F.avg(F.when(F.col("Returned") == "Yes", 1.0).otherwise(0.0)), 3).alias("return_rate")))
summary.orderBy("RegionClean", "Year").show()

(oj.withColumn("Year", F.year("OrderDate"))
   .groupBy("RegionClean").pivot("Year").agg(F.round(F.sum("NetRevenue"), 0))
   .orderBy("RegionClean").show())

In [ ]:
# Q4.1 — rank reps within region
rep_rev = (oj.groupBy("RegionClean", "RepID")
             .agg(F.sum("NetRevenue").alias("rev")))
w = Window.partitionBy("RegionClean").orderBy(F.desc("rev"))
(rep_rev.withColumn("row_number", F.row_number().over(w))
        .withColumn("rank", F.rank().over(w))
        .withColumn("dense_rank", F.dense_rank().over(w))
        .filter("row_number <= 3").show())

# Q4.2 — top 3 products per category
prod_rev = oj.groupBy("Category", "ProductName").agg(F.sum("NetRevenue").alias("rev"))
w2 = Window.partitionBy("Category").orderBy(F.desc("rev"))
prod_rev.withColumn("rn", F.row_number().over(w2)).filter("rn <= 3").show(20, False)

In [ ]:
# Q4.3 / Q4.4 — moving average and MoM growth
monthly = (o.withColumn("Month", F.trunc("OrderDate", "month"))
             .groupBy("Month").agg(F.sum("NetRevenue").alias("rev"))
             .orderBy("Month"))
wm = Window.orderBy("Month").rowsBetween(-2, 0)
wl = Window.orderBy("Month")
(monthly
   .withColumn("ma3", F.round(F.avg("rev").over(wm), 2))
   .withColumn("prev", F.lag("rev").over(wl))
   .withColumn("mom_pct", F.round((F.col("rev") - F.col("prev")) / F.col("prev") * 100, 1))
   .show(12))

# Q4.5 — cumulative
wc = Window.partitionBy("RegionClean").orderBy("OrderDate") \
           .rowsBetween(Window.unboundedPreceding, Window.currentRow)
o.withColumn("cum_rev", F.sum("NetRevenue").over(wc)) \
 .select("RegionClean", "OrderDate", "NetRevenue", "cum_rev").show(5)

In [ ]:
# Q6.1 — when/otherwise beats a UDF
bucketed = o.withColumn("ShipBucket",
    F.when(F.col("ShipDays").isNull(), "Unknown")
     .when(F.col("ShipDays") <= 3, "1-3")
     .when(F.col("ShipDays") <= 6, "4-6")
     .when(F.col("ShipDays") <= 9, "7-9")
     .otherwise("10+"))
bucketed.groupBy("ShipBucket").count().orderBy("ShipBucket").show()

# The UDF version serialises every row to a Python process and back.
# when/otherwise stays inside the JVM and is visible to Catalyst.
# Interview answer: "UDFs are opaque to the optimiser — no predicate pushdown,
# no code generation. Reach for them only when no built-in exists."